In [2]:
import subprocess

result = subprocess.run(
    ["python", "-m", "src.retinocare.models.train",
     "--config", "configs/train_config.yaml",
     "--model", "resnet18"],
    cwd="..", capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

Using device: cpu
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\user/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
Epoch 1/30 | train_loss=1.5292 train_f1=0.4875 | val_loss=1.4082 val_f1=0.5571
  -> new best val_f1=0.5571, saved to models\resnet18.pt
Epoch 2/30 | train_loss=1.3270 train_f1=0.6270 | val_loss=1.2601 val_f1=0.6168
  -> new best val_f1=0.6168, saved to models\resnet18.pt
Epoch 3/30 | train_loss=1.2444 train_f1=0.6431 | val_loss=1.1728 val_f1=0.6473
  -> new best val_f1=0.6473, saved to models\resnet18.pt
Epoch 4/30 | train_loss=1.1626 train_f1=0.6762 | val_loss=1.1466 val_f1=0.6416
Epoch 5/30 | train_loss=1.1356 train_f1=0.6742 | val_loss=1.0981 val_f1=0.6255
Epoch 5: unfreezing backbone for full fine-tuning
Epoch 6/30 | train_loss=0.9665 train_f1=0.7008 | val_loss=0.9666 val_f1=0.7270
  -> new best val_f1=0.7270, saved to models\resnet18.pt
Epoch 7/30 | train_loss=0.7890 train_f1=0.7745 | val_loss=0.9298 val_f1=0.7488
  -> new b

In [4]:
import sys
sys.path.append("..")

import yaml, torch
import torch.nn.functional as F
from src.retinocare.data.dataset import get_dataloaders
from src.retinocare.models.resnet_transfer import build_resnet18
from src.retinocare.evaluation.metrics import compute_classification_report, plot_confusion_matrix, expected_calibration_error

with open("../configs/train_config.yaml") as f:
    config = yaml.safe_load(f)

config["data"]["train_csv"] = "../" + config["data"]["train_csv"]
config["data"]["val_csv"] = "../" + config["data"]["val_csv"]
config["data"]["test_csv"] = "../" + config["data"]["test_csv"]
config["data"]["image_dir"] = "../" + config["data"]["image_dir"]

_, _, test_dl = get_dataloaders(config)

model = build_resnet18(num_classes=5, freeze_backbone=False)
model.load_state_dict(torch.load("../models/resnet18.pt", map_location="cpu"))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in test_dl:
        logits = model(images)
        probs = F.softmax(logits, dim=1)
        all_preds.extend(logits.argmax(dim=1).tolist())
        all_labels.extend(labels.tolist())
        all_probs.extend(probs.tolist())

import numpy as np
all_probs = np.array(all_probs)

SEVERITY_LABELS = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
report = compute_classification_report(all_labels, all_preds, SEVERITY_LABELS)
print("Weighted F1:", report["weighted avg"]["f1-score"])

plot_confusion_matrix(all_labels, all_preds, SEVERITY_LABELS, "../docs/resnet18_confusion_matrix.png")

ece = expected_calibration_error(all_probs, all_labels, n_bins=10)
print(f"Expected Calibration Error: {ece:.4f}")

c:\ProgramData\anaconda3\envs\retinocare-ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Weighted F1: 0.7951825885836976
Expected Calibration Error: 0.0547


In [ ]:
result = subprocess.run(
    ["python", "-m", "src.retinocare.models.train",
     "--config", "configs/train_config.yaml",
     "--model", "efficientnet_b0"],
    cwd="..", capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
import sys
sys.path.append("..")

import yaml, torch
import torch.nn.functional as F
from src.retinocare.data.dataset import get_dataloaders
from src.retinocare.models.resnet_transfer import build_efficientnet_b0
from src.retinocare.evaluation.metrics import compute_classification_report, plot_confusion_matrix, expected_calibration_error

with open("../configs/train_config.yaml") as f:
    config = yaml.safe_load(f)

config["data"]["train_csv"] = "../" + config["data"]["train_csv"]
config["data"]["val_csv"] = "../" + config["data"]["val_csv"]
config["data"]["test_csv"] = "../" + config["data"]["test_csv"]
config["data"]["image_dir"] = "../" + config["data"]["image_dir"]

_, _, test_dl = get_dataloaders(config)

model = build_efficientnet_b0(num_classes=5, freeze_backbone=False)
model.load_state_dict(torch.load("../models/efficientnet_b0.pt", map_location="cpu"))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in test_dl:
        logits = model(images)
        probs = F.softmax(logits, dim=1)
        all_preds.extend(logits.argmax(dim=1).tolist())
        all_labels.extend(labels.tolist())
        all_probs.extend(probs.tolist())

import numpy as np
all_probs = np.array(all_probs)

SEVERITY_LABELS = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
report = compute_classification_report(all_labels, all_preds, SEVERITY_LABELS)
print("Weighted F1:", report["weighted avg"]["f1-score"])

plot_confusion_matrix(all_labels, all_preds, SEVERITY_LABELS, "../docs/efficientnet_b0_confusion_matrix.png")

ece = expected_calibration_error(all_probs, all_labels, n_bins=10)
print(f"Expected Calibration Error: {ece:.4f}")